In [1]:
import numpy as np
import torch
import gymnasium as gym
from torch import nn
from tianshou.data import Collector, VectorReplayBuffer, Batch
from tianshou.env import DummyVectorEnv
from tianshou.policy import DQNPolicy
from tianshou.trainer import OffpolicyTrainer
from dengue_envs.envs.dengue_diagnostics import DengueDiagnosticsEnv
from dengue_envs.envs.cnn_wrapper import *

C:\Users\segun\AppData\Local\pypoetry\Cache\virtualenvs\dengue-diagnostics-env-Ra5buD89-py3.12\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


In [2]:
env = DengueDiagnosticsEnv(epilength=60, size=400)

In [3]:
import torch
from torch import nn
from tianshou.data import Batch # Necessário para a checagem no forward

class DengueCNN(nn.Module):
    """
    Rede Totalmente Convolucional (FCN) que processa um tensor de entrada
    e retorna um mapa de Q-valores com 6 canais.

    Entrada: (Batch, 4, 400, 400)
    Saída: (Batch, 6, 400, 400)
    """
    def __init__(self, input_shape):
        super().__init__()
        self.input_shape = input_shape  # (4, 400, 400)

        # A rede agora é uma sequência de camadas convolucionais
        # que preservam a dimensão espacial (usando padding)
        self.fcn = nn.Sequential(
            # Bloco 1: Extrai features iniciais
            nn.Conv2d(in_channels=input_shape[0], out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Bloco 2: Aprofunda a análise das features
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            # Camada de Saída: A "mágica" acontece aqui
            # Uma convolução 1x1 para transformar os 64 canais de features
            # em 6 canais de Q-valores, sem alterar a altura e largura.
            nn.Conv2d(in_channels=64, out_channels=6, kernel_size=1, stride=1, padding=0)
        )

    def forward(self, obs, state=None, info={}):
        # --- Lógica de pré-processamento da observação (pode manter) ---
        if isinstance(obs, Batch):
            obs = obs.obs
        if not isinstance(obs, torch.Tensor):
            obs = torch.tensor(obs, dtype=torch.float32)

        device = next(self.parameters()).device
        obs = obs.to(device)

        if obs.ndim == 3: # Se receber uma única observação, adiciona a dimensão do batch
            obs = obs.unsqueeze(0)

        # --- Passa a observação pela rede totalmente convolucional ---
        q_value_map = self.fcn(obs)

        return q_value_map, state

In [11]:
class DengueWrapper(gym.ObservationWrapper):
    """Convert dict observation to 4-channel tensor"""
    def __init__(self, env):
        super().__init__(env)
        # Assuming env.unwrapped.size is defined for H and W
        world_size = self.unwrapped.size
        self.observation_space = gym.spaces.Box(
            low=0, high=3, # Max value can be higher if status codes go beyond 2
            shape=(4, world_size, world_size),  # Channels-first format
            dtype=np.float32
        )
        self._world_size = world_size


    def observation(self, obs_dict):
        tensor = np.zeros((4, self._world_size, self._world_size), dtype=np.float32)

        # Channel 0: Clinical diagnostics
        for case in obs_dict.get('clinical_diagnostic', []):
            x, y, diag = case
            if 0 <= x < self._world_size and 0 <= y < self._world_size:
                tensor[0, x, y] = diag + 1

        # Channel 1: TestD status
        for case_id, status in obs_dict.get('testd', []):
            x, y = self.unwrapped.get_case_xy(case_id)
            if 0 <= x < self._world_size and 0 <= y < self._world_size:
                tensor[1, x, y] = status + 1 # status 0-3 encoded as 1-4

        # Channel 2: TestC status
        for case_id, status in obs_dict.get('testc', []):
            x, y = self.unwrapped.get_case_xy(case_id)
            if 0 <= x < self._world_size and 0 <= y < self._world_size:
                tensor[2, x, y] = status + 1 # status 0-3 encoded as 1-4

        # Channel 3: Active case mask
        current_t = self.unwrapped.t
        for case_idx_in_df, case_data in self.unwrapped.obs_cases.iterrows():
            if case_data.t == current_t: # A case that appeared at the current timestep
                 x, y = int(case_data.x), int(case_data.y)
                 if 0 <= x < self._world_size and 0 <= y < self._world_size:
                    tensor[3, x, y] = 1.0

        return tensor

In [23]:
# testing the CNN
cnn = DengueCNN(input_shape=(4, 400, 400))
obs = env.reset()
obs = DengueWrapper(env).observation(obs)


AttributeError: 'tuple' object has no attribute 'get'